# NevoScan

## Эксперимент 5: Исследование стратегий использования масок

В данном исследовании мы сравниваем три стратегии использования масок сегментации:

- **Стратегия A**: маска как 4-й канал (R, G, B, mask). Продолжение Эксп.4, честное сравнение с fn_weight=3.0
- **Стратегия B**: masked input. Маска вырезает аннотированные области, фон обнуляется (R×mask, G×mask, B×mask). Модель видит только те локальные зоны, где расположены признаки из сводной маски. Это могут быть небольшие разрозненные участки изображения, а не весь невус. Соответственно, сводная маска показывает не границы родинки целиком, а только те пиксели, где находятся аннотированные патологические структуры.
- **Стратегия C**: dual-stream. Два параллельных потока (RGB + маска), конкатенация признаков

**Данные:** Derm7pt + ISIC Task 2

**Val/Test:** Derm7pt

**Backbone:** EfficientNet-B3

Мы стараемся приблизиться к результатам Kawahara et al. (2018), многомодальная архитектура для 7-point checklist

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')

### 1. Монтируем Google Drive и скачиваем данные


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Derm7pt
os.makedirs('/content/dataset', exist_ok=True)
if not os.path.exists('/content/dataset/release_v0'):
    os.system('unzip -q "/content/drive/MyDrive/Диплом/практика_преддипломная/derm7pt.zip" '
              '-d "/content/dataset"')
    print('✓ Derm7pt распакован')

# ISIC Task 2
os.makedirs('/content/isic', exist_ok=True)
if not os.path.exists('/content/isic/images'):
    os.system('wget -q --show-progress -O /content/isic/images.zip '
              'https://isic-challenge-data.s3.amazonaws.com/2018/'
              'ISIC2018_Task1-2_Training_Input.zip')
    os.system('unzip -q /content/isic/images.zip -d /content/isic/tmp')
    os.system('mv "/content/isic/tmp/ISIC2018_Task1-2_Training_Input" /content/isic/images')
    os.system('rm -rf /content/isic/images.zip /content/isic/tmp')
    print('✓ ISIC images готовы')

if not os.path.exists('/content/isic/masks'):
    os.system('wget -q --show-progress -O /content/isic/masks.zip '
              'https://isic-challenge-data.s3.amazonaws.com/2018/'
              'ISIC2018_Task2_Training_GroundTruth_v3.zip')
    os.system('unzip -q /content/isic/masks.zip -d /content/isic/tmp')
    os.system('mv "/content/isic/tmp/ISIC2018_Task2_Training_GroundTruth_v3" /content/isic/masks')
    os.system('rm -rf /content/isic/masks.zip /content/isic/tmp')
    print('✓ ISIC masks готовы')

### 2. Импорты и конфигурация

Все гиперпараметры идентичны Эксп.4 для честного сравнения.

Результаты Эксп.1-4 представлены в таблице:
| Эксп. | F1 | AUC |
|---|---|---|
| Эксп.1 baseline | 0.499 | 0.703 |
| Эксп.2 ISIC 3ch | 0.391 | 0.533 |
| Эксп.3 ISIC 4ch | 0.385 | 0.538 |
| Эксп.4 Derm+ISIC | **0.514** | **0.737** |

In [ ]:
import glob, json, shutil
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score,
    recall_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Пути идентичны Эксп.2-4
DERM_BASE  = '/content/dataset/release_v0'
DERM_IMG   = os.path.join(DERM_BASE, 'images')
DERM_META  = os.path.join(DERM_BASE, 'meta/meta.csv')
DERM_TRAIN = os.path.join(DERM_BASE, 'meta/train_indexes.csv')
DERM_VAL   = os.path.join(DERM_BASE, 'meta/valid_indexes.csv')
DERM_TEST  = os.path.join(DERM_BASE, 'meta/test_indexes.csv')
ISIC_IMG   = '/content/isic/images'
ISIC_MASKS = '/content/isic/masks'
SAVE_DIR   = '/content/drive/MyDrive/Диплом/models'
os.makedirs(SAVE_DIR, exist_ok=True)

# Гиперпараметры идентичны Эксп.4
IMG_SIZE     = 300
BATCH_SIZE   = 16
NUM_EPOCHS   = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-3
THRESHOLD    = 0.4
FOCAL_ALPHA  = 0.25
FOCAL_GAMMA  = 2.0
FN_WEIGHT    = 3.0 # штраф за false negative

FEATURES = [
    'pigment_network', 'streaks', 'pigmentation',
    'regression_structures', 'dots_and_globules',
    'blue_whitish_veil', 'vascular_structures'
]
FEAT_RU = [
    'Пигм. сеть', 'Полосы', 'Пигментация',
    'Регрессия', 'Точки/Глобулы', 'Бело-гол. вуаль', 'Сос. структуры'
]
ARGENZIANO_WEIGHTS = {
    'pigment_network': 2, 'streaks': 1, 'pigmentation': 1,
    'regression_structures': 1, 'dots_and_globules': 1,
    'blue_whitish_veil': 2, 'vascular_structures': 2,
}
ARGENZIANO_SCORES  = [ARGENZIANO_WEIGHTS[f] for f in FEATURES]
SUSPICION_THRESHOLD = 3

ISIC_ATTR = {
    'pigment_network':       'pigment_network',
    'streaks':               'streaks',
    'dots_and_globules':     'globules',
    'pigmentation':          None,
    'regression_structures': None,
    'blue_whitish_veil':     None,
    'vascular_structures':   None,
}

print('✓ Конфиг загружен')

### 3. Загрузка данных

Загружаем оба датасета. Train = ISIC (2594) + Derm7pt train (413) = 3007 изображений.

Val/Test = только Derm7pt (официальный сплит).

In [ ]:
def map_label(text):
    if pd.isna(text): return 0
    return 0 if str(text).lower().strip() in ['absent', 'regular', 'typical'] else 1

# Derm7pt
df_meta = pd.read_csv(DERM_META)
df_derm = df_meta.copy()
for feat in FEATURES:
    df_derm[feat] = df_derm[feat].apply(map_label)

all_files = glob.glob(os.path.join(DERM_IMG, '**/*'), recursive=True)
path_map  = {f.lower(): f for f in all_files if os.path.isfile(f)}
df_derm['full_path'] = df_derm['derm'].apply(
    lambda x: path_map.get(os.path.join(DERM_IMG, x).lower()))
df_derm['mask_path'] = None
df_derm['source']    = 'derm7pt'
df_derm = df_derm[df_derm['full_path'].notna()].reset_index(drop=True)

train_idx = pd.read_csv(DERM_TRAIN).iloc[:, 0].values
val_idx   = pd.read_csv(DERM_VAL).iloc[:, 0].values
derm_train = df_derm.iloc[train_idx].reset_index(drop=True)
val_df     = df_derm.iloc[val_idx].reset_index(drop=True)
print(f'Derm7pt — train: {len(derm_train)} | val: {len(val_df)}')

# ISIC Task 2
os.makedirs('/content/isic/combined_masks', exist_ok=True)

def get_isic_label(img_id, feat):
    attr = ISIC_ATTR.get(feat)
    if attr is None: return -1
    mf = os.path.join(ISIC_MASKS, f'{img_id}_attribute_{attr}.png')
    if not os.path.exists(mf): return 0
    return 1 if np.array(Image.open(mf).convert('L')).max() > 0 else 0

def build_combined_mask(img_id):
    combined = None
    for attr in ['pigment_network', 'negative_network', 'streaks',
                 'milia_like_cysts', 'globules']:
        mf = os.path.join(ISIC_MASKS, f'{img_id}_attribute_{attr}.png')
        if os.path.exists(mf):
            m = np.array(Image.open(mf).convert('L'))
            combined = m if combined is None else np.maximum(combined, m)
    if combined is None: return None
    out = f'/content/isic/combined_masks/{img_id}_combined.png'
    Image.fromarray(combined).save(out)
    return out

isic_files = sorted([f for f in os.listdir(ISIC_IMG) if f.endswith('.jpg')])
records = []
for fname in tqdm(isic_files, desc='Парсим ISIC'):
    img_id = os.path.splitext(fname)[0]
    row = {'full_path': os.path.join(ISIC_IMG, fname), 'source': 'isic'}
    for feat in FEATURES:
        row[feat] = get_isic_label(img_id, feat)
    row['mask_path'] = build_combined_mask(img_id)
    records.append(row)
isic_df = pd.DataFrame(records)
print(f'ISIC — {len(isic_df)} записей | масок: {isic_df["mask_path"].notna().sum()}')

# Train = Derm7pt train + ISIC (как в Эксп.4)
train_df = pd.concat([isic_df, derm_train], ignore_index=True)
print(f'\nTrain Эксп.5: {len(train_df)} '
      f'(ISIC={( train_df["source"]=="isic").sum()}, '
      f'Derm7pt={(train_df["source"]=="derm7pt").sum()})')

### 4. Функция потерь с усиленным штрафом за FN

MaskedFocalLoss с fn_weight=3.0

Формула: $\mathcal{L} = \alpha \cdot (1-e^{-BCE})^\gamma \cdot BCE \cdot w(y)$

где $w(y) = 1 + (fn\_weight - 1) \cdot y$

Позитивные примеры штрафуются в 3 раза сильнее. Мы считаем, что пропуск реального признака меланомы клинически опаснее ложной тревоги.

In [ ]:
class MaskedFocalLoss(nn.Module):
    """
    Focal Loss с маскированием меток -1 и усиленным штрафом
    за ложноотрицательные предсказания (false negative).
    """
    def __init__(self, alpha=0.25, gamma=2.0, fn_weight=3.0):
        super().__init__()
        self.a, self.g, self.fn_w = alpha, gamma, fn_weight

    def forward(self, inp, tgt):
        mask  = (tgt >= 0).float()
        tgt_c = tgt.clamp(min=0)
        # Позитивные примеры получают вес fn_weight, негативные — 1.0
        pw    = torch.ones_like(inp) + (self.fn_w - 1) * tgt_c
        bce   = nn.functional.binary_cross_entropy_with_logits(
                    inp, tgt_c, weight=pw, reduction='none')
        fl    = self.a * (1 - torch.exp(-bce)) ** self.g * bce
        n     = mask.sum()
        return (fl * mask).sum() / n if n > 0 else fl.sum() * 0

### 5. Три датасета, по одному на каждую стратегию маски

**Стратегия A:** вход (R, G, B, mask), 4 канала

**Стратегия B:** вход (R×mask, G×mask, B×mask), 3 канала, фон = 0

**Стратегия C:** два отдельных тензора img(3ch) + mask(1ch) для dual-stream модели

In [ ]:
class DatasetStrategyA(Dataset):
    """
    Стратегия А
    """
    def __init__(self, df, is_train=False, sz=300):
        self.df = df.reset_index(drop=True)
        self.is_train, self.sz = is_train, sz
        self.mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        self.std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

    def __len__(self): return len(self.df)

    def _clahe(self, arr):
        try:
            import cv2
            u8  = (arr * 255).astype(np.uint8)
            lab = cv2.cvtColor(u8, cv2.COLOR_RGB2LAB)
            cl  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            lab[:,:,0] = cl.apply(lab[:,:,0])
            return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32)/255.
        except: return arr

    def _quality_aug(self, img):
        if torch.rand(1) > 0.3: return img
        scale = torch.FloatTensor(1).uniform_(0.5, 0.9).item()
        small = max(64, int(self.sz * scale))
        return img.resize((small, small), Image.BILINEAR).resize(
            (self.sz, self.sz), Image.BILINEAR)

    def _load_mask(self, mask_path):
        if pd.notna(mask_path) and mask_path and os.path.exists(str(mask_path)):
            return np.array(Image.open(mask_path).convert('L').resize(
                (self.sz, self.sz), Image.NEAREST), np.float32) / 255.
        return np.zeros((self.sz, self.sz), np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['full_path']).convert('RGB')
        if self.is_train: img = self._quality_aug(img)
        img = img.resize((self.sz, self.sz), Image.BILINEAR)
        arr = np.array(img, np.float32) / 255.
        if self.is_train and torch.rand(1) > 0.5: arr = self._clahe(arr)
        mask = self._load_mask(row.get('mask_path'))
        t4 = torch.from_numpy(
            np.concatenate([arr, mask[:,:,np.newaxis]], axis=2)).permute(2,0,1)
        t4[:3] = (t4[:3] - self.mean) / self.std
        t4[3]  = (t4[3] - 0.5) / 0.5
        if self.is_train:
            if torch.rand(1) > .5: t4 = torch.flip(t4, [2])
            if torch.rand(1) > .5: t4 = torch.flip(t4, [1])
        return t4, torch.tensor(row[FEATURES].values.astype(np.float32))


class DatasetStrategyB(Dataset):
    """
    Стратегия B
    """
    def __init__(self, df, is_train=False, sz=300):
        self.df = df.reset_index(drop=True)
        self.is_train, self.sz = is_train, sz
        self.mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        self.std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

    def __len__(self): return len(self.df)

    def _load_mask(self, mask_path):
        if pd.notna(mask_path) and mask_path and os.path.exists(str(mask_path)):
            m = np.array(Image.open(mask_path).convert('L').resize(
                (self.sz, self.sz), Image.NEAREST), np.float32) / 255.
            # Бинаризуем маску
            return (m > 0.5).astype(np.float32)
        # Если маски нет — маска = 1 везде (оригинал не изменяется)
        return np.ones((self.sz, self.sz), np.float32)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(row['full_path']).convert('RGB').resize(
            (self.sz, self.sz), Image.BILINEAR)
        arr  = np.array(img, np.float32) / 255.
        mask = self._load_mask(row.get('mask_path'))

        # Умножаем каждый канал на маску, фон обнуляется
        masked = arr * mask[:,:,np.newaxis] # (H, W, 3)

        t3 = torch.from_numpy(masked).permute(2, 0, 1) # (3, H, W)
        t3 = (t3 - self.mean) / self.std

        if self.is_train:
            if torch.rand(1) > .5: t3 = torch.flip(t3, [2])
            if torch.rand(1) > .5: t3 = torch.flip(t3, [1])

        return t3, torch.tensor(row[FEATURES].values.astype(np.float32))


class DatasetStrategyC(Dataset):
    """
    Стратегия C
    """
    def __init__(self, df, is_train=False, sz=300):
        self.df = df.reset_index(drop=True)
        self.is_train, self.sz = is_train, sz
        self.mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        self.std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

    def __len__(self): return len(self.df)

    def _load_mask(self, mask_path):
        if pd.notna(mask_path) and mask_path and os.path.exists(str(mask_path)):
            return np.array(Image.open(mask_path).convert('L').resize(
                (self.sz, self.sz), Image.NEAREST), np.float32) / 255.
        return np.zeros((self.sz, self.sz), np.float32)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(row['full_path']).convert('RGB').resize(
            (self.sz, self.sz), Image.BILINEAR)
        arr  = np.array(img, np.float32) / 255.
        mask = self._load_mask(row.get('mask_path'))

        t_img  = torch.from_numpy(arr).permute(2, 0, 1) # (3, H, W)
        t_img  = (t_img - self.mean) / self.std
        t_mask = torch.from_numpy(mask).unsqueeze(0) # (1, H, W)
        t_mask = (t_mask - 0.5) / 0.5

        if self.is_train:
            if torch.rand(1) > .5:
                t_img  = torch.flip(t_img,  [2])
                t_mask = torch.flip(t_mask, [2])
            if torch.rand(1) > .5:
                t_img  = torch.flip(t_img,  [1])
                t_mask = torch.flip(t_mask, [1])

        labels = torch.tensor(row[FEATURES].values.astype(np.float32))
        return t_img, t_mask, labels

### 6. Три архитектуры моделей

**Модель A:** EfficientNet-B3, 4 входных канала (как в Эксп.4)

**Модель B:** EfficientNet-B3, 3 входных канала (стандартная)

**Модель C:** DualStreamEfficientNet — два EfficientNet-B3, конкатенация 1536+1536=3072 признака



In [ ]:
def build_model_A():
    """
    Стратегия A
    """
    m   = timm.create_model('efficientnet_b3', pretrained=True,
                             num_classes=len(FEATURES))
    old = m.conv_stem
    new = nn.Conv2d(4, old.out_channels, old.kernel_size,
                    old.stride, old.padding,
                    bias=old.bias is not None)
    with torch.no_grad():
        new.weight[:,:3] = old.weight
        new.weight[:,3:] = old.weight.mean(dim=1, keepdim=True)
    m.conv_stem = new
    n = sum(p.numel() for p in m.parameters())/1e6
    print(f'Стратегия A: EfficientNet-B3 4ch | {n:.1f}M параметров')
    return m


def build_model_B():
    """
    Стратегия B
    """
    m = timm.create_model('efficientnet_b3', pretrained=True,
                           num_classes=len(FEATURES))
    n = sum(p.numel() for p in m.parameters())/1e6
    print(f'Стратегия B: EfficientNet-B3 3ch masked | {n:.1f}M параметров')
    return m


class DualStreamEfficientNet(nn.Module):
    """
    Стратегия C
    """
    def __init__(self, n_classes=7, dropout=0.3):
        super().__init__()

        # RGB-поток: стандартный EfficientNet-B3
        self.rgb_stream = timm.create_model(
            'efficientnet_b3', pretrained=True, num_classes=0)
        feat_dim = self.rgb_stream.num_features  # 1536

        # Маскирующий поток: EfficientNet-B3 с 1 входным каналом
        self.mask_stream = timm.create_model(
            'efficientnet_b3', pretrained=True, num_classes=0)
        old = self.mask_stream.conv_stem
        new = nn.Conv2d(1, old.out_channels, old.kernel_size,
                        old.stride, old.padding,
                        bias=old.bias is not None)
        with torch.no_grad():
            # Инициализируем как среднее по 3 каналам
            new.weight[:] = old.weight.mean(dim=1, keepdim=True)
        self.mask_stream.conv_stem = new

        # Финальный классификатор
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim * 2, n_classes)
        )

        n = sum(p.numel() for p in self.parameters())/1e6
        print(f'Стратегия C: DualStream EfficientNet-B3 | {n:.1f}M параметров')
        print(f'  RGB stream:  EfficientNet-B3 → {feat_dim}D')
        print(f'  Mask stream: EfficientNet-B3 (1ch) → {feat_dim}D')
        print(f'  Classifier:  Linear({feat_dim*2}→{n_classes})')

    def forward(self, img, mask):
        feat_rgb  = self.rgb_stream(img) # (B, 1536)
        feat_mask = self.mask_stream(mask) # (B, 1536)
        fused     = torch.cat([feat_rgb, feat_mask], dim=1) # (B, 3072)
        return self.classifier(fused) # (B, 7)

### 7. Обучение и оценка

Единая функция run_experiment работает для всех трёх стратегий. Для C используется отдельный forward pass с двумя входами.

In [ ]:
criterion = MaskedFocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA,
                            fn_weight=FN_WEIGHT)


def train_epoch_ab(model, loader, optimizer):
    """Один шаг обучения для стратегий A и B (один тензор входа)."""
    model.train(); total = 0
    for x, labels in tqdm(loader, desc='train', leave=False):
        x, labels = x.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), labels)
        loss.backward(); optimizer.step()
        total += loss.item()
    return total / len(loader)


def train_epoch_c(model, loader, optimizer):
    """Один шаг обучения для стратегии C (два тензора входа)."""
    model.train(); total = 0
    for img, mask, labels in tqdm(loader, desc='train', leave=False):
        img, mask, labels = img.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(img, mask), labels)
        loss.backward(); optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate_ab(model, loader):
    model.eval()
    total = 0; all_probs, all_preds, all_labels = [], [], []
    for x, labels in tqdm(loader, desc='eval', leave=False):
        x, labels = x.to(device), labels.to(device)
        out  = model(x)
        total += criterion(out, labels).item()
        prob = torch.sigmoid(out).cpu().numpy()
        all_probs.append(prob)
        all_preds.append((prob >= THRESHOLD).astype(int))
        all_labels.append(labels.cpu().numpy())
    return _compute_metrics(total/len(loader), all_probs, all_preds, all_labels)


@torch.no_grad()
def evaluate_c(model, loader):
    model.eval()
    total = 0; all_probs, all_preds, all_labels = [], [], []
    for img, mask, labels in tqdm(loader, desc='eval', leave=False):
        img, mask, labels = (img.to(device), mask.to(device), labels.to(device))
        out  = model(img, mask)
        total += criterion(out, labels).item()
        prob = torch.sigmoid(out).cpu().numpy()
        all_probs.append(prob)
        all_preds.append((prob >= THRESHOLD).astype(int))
        all_labels.append(labels.cpu().numpy())
    return _compute_metrics(total/len(loader), all_probs, all_preds, all_labels)


def _compute_metrics(avg_loss, all_probs, all_preds, all_labels):
    probs  = np.vstack(all_probs)
    preds  = np.vstack(all_preds)
    labels = np.vstack(all_labels)
    f1s, aucs = [], []
    for i in range(labels.shape[1]):
        m = labels[:, i] >= 0
        if not m.any(): continue
        f1s.append(f1_score(labels[m,i], preds[m,i], zero_division=0))
        try:    aucs.append(roc_auc_score(labels[m,i], probs[m,i]))
        except: aucs.append(0.)
    return {'loss': avg_loss, 'macro_f1': float(np.mean(f1s)),
            'macro_auc': float(np.mean(aucs)),
            'probs': probs, 'preds': preds, 'labels': labels}


def run_experiment(name, model, train_ds, val_ds, save_path,
                   is_dual_stream=False):
    """Единая функция обучения для всех трёх стратегий."""
    model = model.to(device)
    collate = None
    if is_dual_stream:
        # DataLoader для стратегии C возвращает 3 тензора
        ld_tr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, drop_last=True)
        ld_vl = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2)
        train_fn = train_epoch_c
        eval_fn  = evaluate_c
    else:
        ld_tr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, drop_last=True)
        ld_vl = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2)
        train_fn = train_epoch_ab
        eval_fn  = evaluate_ab

    optimizer = optim.AdamW(model.parameters(), lr=LR,
                            weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                      T_max=NUM_EPOCHS)

    best_loss = float('inf')
    tl_hist, vl_hist, vf_hist = [], [], []

    print(f'\n{"═"*60}')
    print(f'  {name}')
    print(f'  Train: {len(train_ds)} | Val: {len(val_ds)}')
    print(f'  fn_weight={FN_WEIGHT} | threshold={THRESHOLD}')
    print(f'{"═"*60}')

    for ep in range(1, NUM_EPOCHS + 1):
        tl = train_fn(model, ld_tr, optimizer)
        vm = eval_fn(model, ld_vl)
        scheduler.step()
        tl_hist.append(tl); vl_hist.append(vm['loss']); vf_hist.append(vm['macro_f1'])
        print(f'[{name}] ep {ep:02d}/{NUM_EPOCHS} | '
              f'train={tl:.4f}  val={vm["loss"]:.4f}  '
              f'F1={vm["macro_f1"]:.3f}  AUC={vm["macro_auc"]:.3f}')
        if vm['loss'] < best_loss:
            best_loss = vm['loss']
            torch.save(model.state_dict(), save_path)
            print(f'  ✓ Лучшая модель сохранена (ep {ep})')

    model.load_state_dict(torch.load(save_path))
    final = eval_fn(model, ld_vl)

    # Кривые обучения
    _plot_curves(name, tl_hist, vl_hist, vf_hist)
    return model, final


def _plot_curves(name, tl, vl, vf):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'Кривые обучения — {name}', fontsize=12)
    axes[0].plot(tl, label='Train', marker='o', ms=3, color='#4C72B0')
    axes[0].plot(vl, label='Val',   marker='s', ms=3, ls='--', color='#C44E52')
    best_ep = int(np.argmin(vl))
    axes[0].axvline(best_ep, color='gray', ls=':', alpha=.7,
                    label=f'Best ep {best_ep+1}')
    axes[0].set_title('Focal Loss'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(vf, color='#55A868', marker='o', ms=3)
    axes[1].axhline(0.5,   ls='--', color='red',    alpha=.6, label='Baseline 0.5')
    axes[1].axhline(0.512, ls=':',  color='orange', alpha=.6, label='Эксп.4 (0.512)')
    axes[1].set_title('Macro F1 (Val)'); axes[1].legend(); axes[1].grid(alpha=.3)
    plt.tight_layout()
    safe = name.replace(' ', '_').replace('/', '_')
    plt.savefig(f'/content/curves_{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()

### 8. Подсчёт баллов по шкале Argenziano

Балл Argenziano вычисляется как взвешенная сумма предсказаний:

$S = \sum_{i=1}^{7} w_i \cdot \hat{y}_i$, где $w_i$ — вес признака (2 или 1 балл)

При $S \geq 3$ подозрение на меланому

In [ ]:
def compute_argenziano_score(feat_probs, threshold=THRESHOLD):
    preds = (feat_probs >= threshold).astype(int)
    score = int(np.dot(preds, ARGENZIANO_SCORES))
    found = [FEAT_RU[i] for i, p in enumerate(preds) if p == 1]
    return score, int(score >= SUSPICION_THRESHOLD), found


def argenziano_summary(m, name):
    probs  = m['probs']
    labels = m['labels']
    N      = len(probs)
    scores = np.array([compute_argenziano_score(probs[i])[0] for i in range(N)])
    verdicts = (scores >= SUSPICION_THRESHOLD).astype(int)

    gt_scores = np.array([
        sum(ARGENZIANO_WEIGHTS[f] * int(labels[i,j] == 1)
            for j, f in enumerate(FEATURES) if labels[i,j] >= 0)
        for i in range(N)
    ])
    gt_verdicts = (gt_scores >= SUSPICION_THRESHOLD).astype(int)

    cm = confusion_matrix(gt_verdicts, verdicts)
    print(f'\n  [{name}] Argenziano: подозрительных = '
          f'{verdicts.sum()} ({100*verdicts.mean():.1f}%)')
    if cm.shape == (2,2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp/(tp+fn) if (tp+fn)>0 else 0
        spec = tn/(tn+fp) if (tn+fp)>0 else 0
        print(f'  Чувствительность: {sens:.3f} | Специфичность: {spec:.3f}')
    return scores, verdicts

### 9. Вывод метрик

Выводим метрики по каждому из 7 признаков + Macro F1/AUC.

In [ ]:
def print_metrics(name, m):
    print(f'\n{"="*62}')
    print(f'  {name}  |  Val = Derm7pt')
    print(f'  Macro F1: {m["macro_f1"]:.4f}  |  Macro AUC: {m["macro_auc"]:.4f}')
    print(f'{"─"*62}')
    print(f'  {"Признак":<26} {"F1":>6} {"Prec":>6} '
          f'{"Recall":>7} {"AUC":>7} {"Pos":>5}')
    print(f'{"─"*62}')
    for i, (feat, name_ru) in enumerate(zip(FEATURES, FEAT_RU)):
        msk = m['labels'][:, i] >= 0
        if not msk.any(): continue
        f1  = f1_score(m['labels'][msk,i], m['preds'][msk,i], zero_division=0)
        pr  = precision_score(m['labels'][msk,i], m['preds'][msk,i], zero_division=0)
        rec = recall_score(m['labels'][msk,i], m['preds'][msk,i], zero_division=0)
        try:    auc = roc_auc_score(m['labels'][msk,i], m['probs'][msk,i])
        except: auc = 0.
        pos = int(m['labels'][msk,i].sum())
        wt  = ARGENZIANO_WEIGHTS[feat]
        star = ' ★' if wt == 2 else ''
        print(f'  {name_ru:<26} {f1:>6.3f} {pr:>6.3f} '
              f'{rec:>7.3f} {auc:>7.3f} {pos:>5}  {wt}б{star}')
    print(f'{"─"*62}')
    print(f'  {"MACRO AVG":<26} {m["macro_f1"]:>6.3f} {"":>6} '
          f'{"":>7} {m["macro_auc"]:>7.3f}')
    print(f'{"="*62}')

### 10. Запускаем обучение

Стратегии обучаются последовательно, лучшие веса сохраняются автоматически.

In [ ]:
print('\n' + '▓'*62)
print('  ЭКСПЕРИМЕНТ 5: СРАВНЕНИЕ СТРАТЕГИЙ МАСОК')
print('▓'*62)

# Стратегия A
modelA, metricsA = run_experiment(
    name      = 'Стратегия A: 4ch (RGB + маска)',
    model     = build_model_A(),
    train_ds  = DatasetStrategyA(train_df, is_train=True),
    val_ds    = DatasetStrategyA(val_df,   is_train=False),
    save_path = '/content/best_exp5A.pth',
)
print_metrics('СТРАТЕГИЯ A — 4ch (RGB + маска)', metricsA)
scoresA, _ = argenziano_summary(metricsA, 'Стратегия A')

# Стратегия B
modelB, metricsB = run_experiment(
    name      = 'Стратегия B: Masked Input (RGB × маска)',
    model     = build_model_B(),
    train_ds  = DatasetStrategyB(train_df, is_train=True),
    val_ds    = DatasetStrategyB(val_df,   is_train=False),
    save_path = '/content/best_exp5B.pth',
)
print_metrics('СТРАТЕГИЯ B — Masked Input', metricsB)
scoresB, _ = argenziano_summary(metricsB, 'Стратегия B')

# Стратегия C
modelC, metricsC = run_experiment(
    name         = 'Стратегия C: Dual-Stream',
    model        = DualStreamEfficientNet(n_classes=len(FEATURES)),
    train_ds     = DatasetStrategyC(train_df, is_train=True),
    val_ds       = DatasetStrategyC(val_df,   is_train=False),
    save_path    = '/content/best_exp5C.pth',
    is_dual_stream=True,
)
print_metrics('СТРАТЕГИЯ C — Dual-Stream', metricsC)
scoresC, _ = argenziano_summary(metricsC, 'Стратегия C')

### 11. Сводная таблица всех экспериментов


In [ ]:
# Результаты Эксп.1-4 (из предыдущего ноутбука)
exp1 = {'macro_f1': 0.499, 'macro_auc': 0.703}
exp2 = {'macro_f1': 0.391, 'macro_auc': 0.533}
exp3 = {'macro_f1': 0.385, 'macro_auc': 0.538}
exp4 = {'macro_f1': 0.514, 'macro_auc': 0.737}

all_results = [
    ('Эксп.1  Derm7pt, 3ch, baseline',           exp1),
    ('Эксп.2  ISIC,    3ch, без маски',           exp2),
    ('Эксп.3  ISIC,    4ch, маска как канал',     exp3),
    ('Эксп.4  Derm+ISIC, 4ch, CLAHE+aug',         exp4),
    ('Эксп.5A Derm+ISIC, 4ch, fn_weight=3',       metricsA),
    ('Эксп.5B Derm+ISIC, masked input, fn_w=3',  metricsB),
    ('Эксп.5C Derm+ISIC, dual-stream, fn_w=3',   metricsC),
]

baseline_f1 = exp1['macro_f1']
print(f'\n{"="*70}')
print(f'  СВОДНАЯ ТАБЛИЦА — ВСЕ ЭКСПЕРИМЕНТЫ')
print(f'  Val = Derm7pt (203 изображения), threshold = {THRESHOLD}')
print(f'{"="*70}')
print(f'  {"Конфигурация":<50} {"F1":>6} {"AUC":>7} {"ΔF1":>7}')
print(f'  {"─"*67}')
for conf_name, m in all_results:
    f1  = m['macro_f1']
    auc = m['macro_auc']
    delta = f1 - baseline_f1
    sign  = '+' if delta >= 0 else ''
    print(f'  {conf_name:<50} {f1:>6.3f} {auc:>7.3f} {sign}{delta:>6.3f}')
print(f'{"="*70}')

# Визуализация сравнения
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Сравнение всех экспериментов (Val = Derm7pt)', fontsize=13)

names = [r[0].split('  ')[0] for r in all_results]
f1s   = [r[1]['macro_f1']  for r in all_results]
aucs  = [r[1]['macro_auc'] for r in all_results]
colors = ['#4C72B0','#4C72B0','#55A868','#55A868',
          '#C44E52','#DD8452','#937860']

for ax, vals, title, ylab in [
    (axes[0], f1s,  'Macro F1',      'F1'),
    (axes[1], aucs, 'Macro AUC-ROC', 'AUC'),
]:
    bars = ax.bar(names, vals, color=colors, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.005, f'{v:.3f}',
                ha='center', va='bottom', fontsize=8)
    ax.axhline(0.5, ls='--', color='red', alpha=.5, label='Baseline 0.5')
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel(ylab); ax.set_title(title)
    ax.legend(); ax.grid(axis='y', alpha=.3)
    ax.set_ylim(0.3, 0.85)

plt.tight_layout()
plt.savefig('/content/exp5_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Argenziano scores сравнение
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.arange(-0.5, 11.5, 1)
for scores, label, color in [
    (scoresA, 'Стратегия A (4ch)', '#C44E52'),
    (scoresB, 'Стратегия B (masked)', '#DD8452'),
    (scoresC, 'Стратегия C (dual-stream)', '#937860'),
]:
    ax.hist(scores, bins=bins, alpha=0.5, label=label,
            color=color, density=True)
ax.axvline(SUSPICION_THRESHOLD - 0.5, color='red', ls='--', lw=2,
           label=f'Порог подозрения ({SUSPICION_THRESHOLD}б)')
ax.set_xlabel('Балл по шкале Argenziano')
ax.set_ylabel('Плотность')
ax.set_title('Распределение баллов Argenziano — Эксп.5 (три стратегии)')
ax.set_xticks(range(0, 11))
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig('/content/exp5_argenziano.png', dpi=150, bbox_inches='tight')
plt.show()

### 12. Сохранение на Google Drive

Сохраняем веса моделей, графики и JSON с результатами.

In [ ]:
files = [
    'best_exp5A.pth', 'best_exp5B.pth', 'best_exp5C.pth',
    'exp5_summary.png', 'exp5_argenziano.png',
]
print('\nСохраняем на Google Drive...')
for f in files:
    src = f'/content/{f}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SAVE_DIR, f))
        print(f'  ✓ {f}')

# JSON с результатами для сводной таблицы диплома
results_json = {
    'exp5A': {'macro_f1': round(metricsA['macro_f1'],4),
              'macro_auc': round(metricsA['macro_auc'],4)},
    'exp5B': {'macro_f1': round(metricsB['macro_f1'],4),
              'macro_auc': round(metricsB['macro_auc'],4)},
    'exp5C': {'macro_f1': round(metricsC['macro_f1'],4),
              'macro_auc': round(metricsC['macro_auc'],4)},
}
with open(os.path.join(SAVE_DIR, 'results_exp5.json'), 'w',
          encoding='utf-8') as f_out:
    json.dump(results_json, f_out, ensure_ascii=False, indent=2)
print('  ✓ results_exp5.json')

best_strategy = max(
    [('A', metricsA), ('B', metricsB), ('C', metricsC)],
    key=lambda x: x[1]['macro_f1']
)
print(f'\n✓ Лучшая стратегия: {best_strategy[0]} '
      f'(F1={best_strategy[1]["macro_f1"]:.4f}, '
      f'AUC={best_strategy[1]["macro_auc"]:.4f})')
print('Используем эту стратегию в Эксп.6 (MultiHead 8 голов)')